# Evaluate Fine-tuned Progressive S2 (Custom+NTUT)

Load ft_best.pt cho moi seed, chay evaluation tren custom val set.
Output: P, R, F1, mAP@0.5, mAP@[.5:.95] cho tung seed.

In [1]:
import os, json, random, gc
import numpy as np
import torch
import torch.nn as nn
import cv2
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from ultralytics import YOLO
from ultralytics.nn.modules import C2f, Conv, Detect

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

Device: cuda


In [2]:
# =====================================================================
# Paths - auto-detect local (Windows) vs GPU server (Linux)
# =====================================================================
import platform
if platform.system() == 'Windows':
    BASE_DIR = r'E:\FPTU\AIP491'
    NUM_WORKERS = 0   # Windows multiprocess DataLoader hay loi
else:
    BASE_DIR = '/root/AIP491'
    NUM_WORKERS = 4

CUSTOM_DIR   = os.path.join(BASE_DIR, 'custom_data')
BACKBONE_DIR = os.path.join(BASE_DIR, 'backbones')
FT_OUT_DIR   = os.path.join(BASE_DIR, 'Finetune', 'outputs', 'progressive_s2_custom_ntut')

RGB_BB_PATH  = os.path.join(BACKBONE_DIR, 'llvip_rgb_best.pt')
THR_BB_PATH  = os.path.join(BACKBONE_DIR, 'llvip_thermal_best.pt')

IMG_SIZE     = 640
BATCH_SIZE   = 16
SEEDS        = [42, 777, 123]
CONF_THRESH  = 0.001   # low thresh for mAP computation
IOU_THRESH   = 0.5     # NMS IoU

print(f'Platform: {platform.system()}')
print(f'BASE_DIR: {BASE_DIR}')
print(f'Custom data: {os.path.exists(os.path.join(CUSTOM_DIR, "annotation.json"))}')
print(f'FT checkpoints: {os.path.exists(FT_OUT_DIR)}')

Platform: Windows
BASE_DIR: E:\FPTU\AIP491
Custom data: True
FT checkpoints: True


In [3]:
# =====================================================================
# Model class (same as training)
# =====================================================================
class RGBTFusionDetector(nn.Module):
    EXTRACT_LAYERS = {4: 64, 6: 128, 9: 256}

    def __init__(self, rgb_backbone, thermal_backbone, nc=1, freeze_backbones=False):
        super().__init__()
        self.rgb_stream     = rgb_backbone
        self.thermal_stream = thermal_backbone

        self.fuse_p3 = nn.Sequential(
            nn.Conv2d(128, 128, 1, bias=False), nn.BatchNorm2d(128), nn.SiLU())
        self.fuse_p4 = nn.Sequential(
            nn.Conv2d(256, 256, 1, bias=False), nn.BatchNorm2d(256), nn.SiLU())
        self.fuse_p5 = nn.Sequential(
            nn.Conv2d(512, 512, 1, bias=False), nn.BatchNorm2d(512), nn.SiLU())
        self.upsample   = nn.Upsample(scale_factor=2, mode='nearest')
        self.td_c2f_p4  = C2f(512 + 256, 256, n=1, shortcut=False)
        self.td_c2f_p3  = C2f(256 + 128, 128, n=1, shortcut=False)
        self.bu_conv_p4 = Conv(128, 128, 3, 2)
        self.bu_c2f_p4  = C2f(128 + 256, 256, n=1, shortcut=False)
        self.bu_conv_p5 = Conv(256, 256, 3, 2)
        self.bu_c2f_p5  = C2f(256 + 512, 512, n=1, shortcut=False)
        self.detect     = Detect(nc=nc, ch=(128, 256, 512))
        self.detect.stride = torch.tensor([8., 16., 32.])

    def _extract(self, stream, x):
        feats = {}
        for i, layer in enumerate(stream):
            x = layer(x)
            if i in self.EXTRACT_LAYERS:
                feats[i] = x
        return feats

    def forward(self, rgb, thermal):
        rf = self._extract(self.rgb_stream, rgb)
        tf = self._extract(self.thermal_stream, thermal)
        p3 = self.fuse_p3(torch.cat([rf[4], tf[4]], dim=1))
        p4 = self.fuse_p4(torch.cat([rf[6], tf[6]], dim=1))
        p5 = self.fuse_p5(torch.cat([rf[9], tf[9]], dim=1))
        p4_td  = self.td_c2f_p4(torch.cat([self.upsample(p5), p4], dim=1))
        p3_out = self.td_c2f_p3(torch.cat([self.upsample(p4_td), p3], dim=1))
        p4_out = self.bu_c2f_p4(torch.cat([self.bu_conv_p4(p3_out), p4_td], dim=1))
        p5_out = self.bu_c2f_p5(torch.cat([self.bu_conv_p5(p4_out), p5], dim=1))
        return self.detect([p3_out, p4_out, p5_out])


def load_backbone(path):
    src = path if (path and os.path.exists(path)) else 'yolov8n.pt'
    return nn.ModuleList(list(YOLO(src).model.model)[:10]).to(device)

print('Model class OK')

Model class OK


In [4]:
# =====================================================================
# Parse custom val set (same as training notebook)
# =====================================================================
def parse_custom_coco(custom_dir, val_ratio=0.2, seed=42):
    ann_path = Path(custom_dir) / 'annotation.json'
    rgb_dir  = Path(custom_dir) / 'rgb'
    thr_dir  = Path(custom_dir) / 'thermal_1'

    with open(str(ann_path)) as f:
        coco = json.load(f)

    id2img = {img['id']: (img['file_name'],
                          float(img.get('width', 1920)),
                          float(img.get('height', 1080)))
              for img in coco['images']}

    valid_cats = {cat['id']: 0 for cat in coco['categories']
                  if any(k in cat['name'].lower() for k in ('person', 'people'))}
    if not valid_cats:
        valid_cats = {1: 0}

    img2boxes = {}
    for ann in coco['annotations']:
        img_id = ann['image_id']
        if ann['category_id'] not in valid_cats or img_id not in id2img:
            continue
        _, iw, ih = id2img[img_id]
        x, y, bw, bh = [float(v) for v in ann['bbox']]
        cx, cy = (x + bw / 2.0) / iw, (y + bh / 2.0) / ih
        nw, nh = bw / iw, bh / ih
        if nw > 0 and nh > 0:
            img2boxes.setdefault(img_id, []).append([0, cx, cy, nw, nh])

    all_samples = []
    for img_id, boxes in img2boxes.items():
        fname = id2img[img_id][0]
        rgb_p = rgb_dir / fname
        thr_p = thr_dir / fname
        if not rgb_p.exists():
            continue
        all_samples.append({
            'rgb_path':   str(rgb_p),
            'thr_path':   str(thr_p) if thr_p.exists() else None,
            'pseudo_thr': not thr_p.exists(),
            'boxes':      boxes,
        })

    rng = random.Random(seed)
    rng.shuffle(all_samples)
    n_val = max(1, int(len(all_samples) * val_ratio))
    val_s = all_samples[:n_val]
    print(f'Val samples: {len(val_s)}')
    return val_s

val_samples = parse_custom_coco(CUSTOM_DIR, val_ratio=0.2, seed=42)

Val samples: 331


In [5]:
# =====================================================================
# Inference + mAP evaluation
# =====================================================================
def nms_numpy(boxes, scores, iou_thresh=0.45):
    if len(boxes) == 0:
        return []
    x1, y1 = boxes[:, 0], boxes[:, 1]
    x2, y2 = boxes[:, 2], boxes[:, 3]
    areas = (x2 - x1) * (y2 - y1)
    order = scores.argsort()[::-1]
    keep = []
    while len(order) > 0:
        i = order[0]
        keep.append(i)
        if len(order) == 1:
            break
        xx1 = np.maximum(x1[i], x1[order[1:]])
        yy1 = np.maximum(y1[i], y1[order[1:]])
        xx2 = np.minimum(x2[i], x2[order[1:]])
        yy2 = np.minimum(y2[i], y2[order[1:]])
        inter = np.maximum(0, xx2 - xx1) * np.maximum(0, yy2 - yy1)
        iou = inter / (areas[i] + areas[order[1:]] - inter + 1e-6)
        inds = np.where(iou <= iou_thresh)[0]
        order = order[inds + 1]
    return keep


def predict_batch(model, rgb_t, thr_t, conf=0.001, nms_iou=0.45):
    """Run inference, return list of (boxes_xyxy, scores) per image."""
    model.eval()
    with torch.no_grad():
        preds = model(rgb_t.to(device), thr_t.to(device))
    # preds shape: (B, 5, 8400) -> transpose to (B, 8400, 5)
    if isinstance(preds, (tuple, list)):
        preds = preds[0]
    preds = preds.cpu().numpy()
    results = []
    for b in range(preds.shape[0]):
        p = preds[b].T  # (8400, 5): cx, cy, w, h, conf
        mask = p[:, 4] > conf
        p = p[mask]
        if len(p) == 0:
            results.append((np.zeros((0, 4)), np.zeros(0)))
            continue
        # cx,cy,w,h -> x1,y1,x2,y2 (normalized 0-1 since input is 640)
        boxes = np.zeros((len(p), 4))
        boxes[:, 0] = (p[:, 0] - p[:, 2] / 2) / IMG_SIZE
        boxes[:, 1] = (p[:, 1] - p[:, 3] / 2) / IMG_SIZE
        boxes[:, 2] = (p[:, 0] + p[:, 2] / 2) / IMG_SIZE
        boxes[:, 3] = (p[:, 1] + p[:, 3] / 2) / IMG_SIZE
        boxes = np.clip(boxes, 0, 1)
        scores = p[:, 4]
        keep = nms_numpy(boxes, scores, nms_iou)
        results.append((boxes[keep], scores[keep]))
    return results


def compute_ap(recalls, precisions):
    """101-point interpolation AP."""
    mrec = np.concatenate(([0.], recalls, [1.]))
    mpre = np.concatenate(([0.], precisions, [0.]))
    for i in range(len(mpre) - 1, 0, -1):
        mpre[i - 1] = max(mpre[i - 1], mpre[i])
    ap = 0.0
    for t in np.linspace(0, 1, 101):
        p = mpre[mrec >= t]
        ap += (p.max() if len(p) > 0 else 0) / 101.0
    return ap


def evaluate_map(all_preds, all_gts, iou_thresh=0.5):
    """Compute P, R, F1, AP at a single IoU threshold."""
    # Collect all detections with image index
    dets = []  # (score, img_idx, box)
    n_gt = 0
    gt_matched = {}  # img_idx -> set of matched gt indices
    
    for img_idx, (pred_boxes, pred_scores) in enumerate(all_preds):
        gt_boxes = all_gts[img_idx]
        n_gt += len(gt_boxes)
        gt_matched[img_idx] = set()
        for j in range(len(pred_boxes)):
            dets.append((pred_scores[j], img_idx, pred_boxes[j]))
    
    if n_gt == 0:
        return 0, 0, 0, 0
    
    # Sort by score descending
    dets.sort(key=lambda x: -x[0])
    
    tp = np.zeros(len(dets))
    fp = np.zeros(len(dets))
    
    for d_idx, (score, img_idx, pred_box) in enumerate(dets):
        gt_boxes = all_gts[img_idx]
        best_iou = 0
        best_gt = -1
        for g_idx, gt_box in enumerate(gt_boxes):
            # Compute IoU
            ix1 = max(pred_box[0], gt_box[0])
            iy1 = max(pred_box[1], gt_box[1])
            ix2 = min(pred_box[2], gt_box[2])
            iy2 = min(pred_box[3], gt_box[3])
            inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)
            a1 = (pred_box[2] - pred_box[0]) * (pred_box[3] - pred_box[1])
            a2 = (gt_box[2] - gt_box[0]) * (gt_box[3] - gt_box[1])
            iou = inter / (a1 + a2 - inter + 1e-6)
            if iou > best_iou:
                best_iou = iou
                best_gt = g_idx
        
        if best_iou >= iou_thresh and best_gt not in gt_matched[img_idx]:
            tp[d_idx] = 1
            gt_matched[img_idx].add(best_gt)
        else:
            fp[d_idx] = 1
    
    cum_tp = np.cumsum(tp)
    cum_fp = np.cumsum(fp)
    recalls = cum_tp / n_gt
    precisions = cum_tp / (cum_tp + cum_fp + 1e-6)
    
    ap = compute_ap(recalls, precisions)
    
    # Best F1 point for P/R reporting
    f1s = 2 * precisions * recalls / (precisions + recalls + 1e-6)
    best_idx = np.argmax(f1s)
    best_p = precisions[best_idx]
    best_r = recalls[best_idx]
    best_f1 = f1s[best_idx]
    
    return best_p, best_r, best_f1, ap

print('Eval functions OK')

Eval functions OK


In [6]:
# =====================================================================
# Val dataset (no augmentation) + collate_fn cho variable-size GT
# =====================================================================
class ValDataset(Dataset):
    def __init__(self, samples, img_size=640):
        self.samples = samples
        self.img_size = img_size
        self.resize = T.Resize((img_size, img_size))
        self.to_tensor = T.ToTensor()

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        rgb_pil = Image.open(s['rgb_path']).convert('RGB')
        if s['pseudo_thr'] or s['thr_path'] is None:
            arr  = np.array(rgb_pil)
            gray = cv2.cvtColor(arr, cv2.COLOR_RGB2GRAY)
            hot  = cv2.applyColorMap(gray, cv2.COLORMAP_HOT)
            thr_pil = Image.fromarray(cv2.cvtColor(hot, cv2.COLOR_BGR2RGB))
        else:
            thr_pil = Image.open(s['thr_path']).convert('L').convert('RGB')

        rgb_t = self.to_tensor(self.resize(rgb_pil))
        thr_t = self.to_tensor(self.resize(thr_pil))

        # GT boxes: normalized cx,cy,w,h -> x1,y1,x2,y2 (normalized)
        gt_boxes = []
        for b in s['boxes']:
            cx, cy, w, h = b[1], b[2], b[3], b[4]
            gt_boxes.append([cx - w/2, cy - h/2, cx + w/2, cy + h/2])

        return rgb_t, thr_t, np.array(gt_boxes, dtype=np.float32)


def val_collate_fn(batch):
    """Custom collate: stack RGB/THR tensors, keep GT as list of arrays."""
    rgbs, thrs, gts = zip(*batch)
    return torch.stack(rgbs), torch.stack(thrs), list(gts)

print('ValDataset + collate OK')

ValDataset + collate OK


In [7]:
# =====================================================================
# Evaluate all 3 seeds
# =====================================================================
val_ds = ValDataset(val_samples, IMG_SIZE)

iou_thresholds = [0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90, 0.95]

all_results = {}
for seed in SEEDS:
    print(f'\n{"="*60}')
    print(f'  Evaluating seed {seed}')
    print(f'{"="*60}')
    
    ckpt_path = os.path.join(FT_OUT_DIR, f'seed_{seed}', 'ft_best.pt')
    if not os.path.exists(ckpt_path):
        print(f'  [SKIP] Checkpoint not found: {ckpt_path}')
        continue
    
    # Build and load model
    rgb_bb = load_backbone(RGB_BB_PATH)
    thr_bb = load_backbone(THR_BB_PATH)
    model = RGBTFusionDetector(rgb_bb, thr_bb, nc=1, freeze_backbones=False).to(device)
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'])
    model.eval()
    
    # Run predictions on val set
    all_preds = []
    all_gts = []
    
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                            num_workers=NUM_WORKERS, collate_fn=val_collate_fn,
                            pin_memory=True)
    
    for rgb_batch, thr_batch, gt_batch in val_loader:
        batch_preds = predict_batch(model, rgb_batch, thr_batch,
                                    conf=CONF_THRESH, nms_iou=IOU_THRESH)
        all_preds.extend(batch_preds)
        all_gts.extend(gt_batch)
    
    # Compute mAP at each IoU threshold
    aps = {}
    for iou_t in iou_thresholds:
        p, r, f1, ap = evaluate_map(all_preds, all_gts, iou_thresh=iou_t)
        aps[f'{iou_t:.2f}'] = ap
        if iou_t == 0.50:
            p50, r50, f1_50 = p, r, f1
    
    map50 = aps['0.50']
    map5095 = np.mean(list(aps.values()))
    
    print(f'  Seed {seed}:  P={p50:.4f}  R={r50:.4f}  F1={f1_50:.4f}')
    print(f'              mAP@0.5={map50:.4f}  mAP@.5:.95={map5095:.4f}')
    
    all_results[seed] = {
        'precision': float(p50),
        'recall': float(r50),
        'f1': float(f1_50),
        'map50': float(map50),
        'map50_95': float(map5095),
        'ap_per_iou': {k: float(v) for k, v in aps.items()},
    }
    
    # Save per-seed result
    eval_path = os.path.join(FT_OUT_DIR, f'seed_{seed}', 'eval_results.json')
    with open(eval_path, 'w') as f:
        json.dump(all_results[seed], f, indent=2)
    print(f'  Saved: {eval_path}')
    
    del model, rgb_bb, thr_bb
    gc.collect()
    torch.cuda.empty_cache()


  Evaluating seed 42
  Seed 42:  P=0.8753  R=0.7951  F1=0.8333
              mAP@0.5=0.8458  mAP@.5:.95=0.4128
  Saved: E:\FPTU\AIP491\Finetune\outputs\progressive_s2_custom_ntut\seed_42\eval_results.json

  Evaluating seed 777
  Seed 777:  P=0.8682  R=0.8068  F1=0.8364
              mAP@0.5=0.8493  mAP@.5:.95=0.4099
  Saved: E:\FPTU\AIP491\Finetune\outputs\progressive_s2_custom_ntut\seed_777\eval_results.json

  Evaluating seed 123
  Seed 123:  P=0.8939  R=0.7845  F1=0.8356
              mAP@0.5=0.8507  mAP@.5:.95=0.4132
  Saved: E:\FPTU\AIP491\Finetune\outputs\progressive_s2_custom_ntut\seed_123\eval_results.json


In [8]:
# =====================================================================
# Summary
# =====================================================================
print('\n' + '=' * 70)
print('  FINETUNE EVALUATION SUMMARY (progressive_s2_custom_ntut)')
print('=' * 70)
print(f'{"Seed":>6}  {"P":>8}  {"R":>8}  {"F1":>8}  {"mAP@.5":>8}  {"mAP@.5:.95":>10}')
print('-' * 60)

best_seed = None
best_map50 = 0
for seed in SEEDS:
    if seed not in all_results:
        continue
    r = all_results[seed]
    print(f'{seed:>6}  {r["precision"]:>8.4f}  {r["recall"]:>8.4f}  {r["f1"]:>8.4f}  {r["map50"]:>8.4f}  {r["map50_95"]:>10.4f}')
    if r['map50'] > best_map50:
        best_map50 = r['map50']
        best_seed = seed

print('-' * 60)
print(f'Best seed: {best_seed} (mAP@0.5 = {best_map50:.4f})')

# Save combined summary
eval_summary = {
    'tag': 'progressive_s2_custom_ntut_eval',
    'seeds': {str(s): all_results[s] for s in all_results},
    'best_seed': best_seed,
    'best_map50': best_map50,
}
summary_path = os.path.join(FT_OUT_DIR, 'eval_summary.json')
with open(summary_path, 'w') as f:
    json.dump(eval_summary, f, indent=2)
print(f'\nSaved: {summary_path}')


  FINETUNE EVALUATION SUMMARY (progressive_s2_custom_ntut)
  Seed         P         R        F1    mAP@.5  mAP@.5:.95
------------------------------------------------------------
    42    0.8753    0.7951    0.8333    0.8458      0.4128
   777    0.8682    0.8068    0.8364    0.8493      0.4099
   123    0.8939    0.7845    0.8356    0.8507      0.4132
------------------------------------------------------------
Best seed: 123 (mAP@0.5 = 0.8507)

Saved: E:\FPTU\AIP491\Finetune\outputs\progressive_s2_custom_ntut\eval_summary.json
